# Algorithm Spot-Checking

This notebook performs spot-checking of different classification algorithms using a reusable pipeline built with `imblearn` and `scikit-learn`.

**Data flow:**
- Input: `kidney_disease_encoded.csv` — dataset with all features already encoded (generated by `Encode_Categorical_Values.ipynb`)
- Normalization is applied **inside the pipeline**, ensuring that `MinMaxScaler` is fitted only on the training data of each fold (no data leakage)
- Balancing (when applicable) is applied only to the training data of each fold

**Approach:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision and Recall
- Classifier and/or sampler can be swapped without changing the pipeline structure

**Balancing strategies evaluated:**
- **No Balancing** — training without any class balancing adjustment
- **SMOTE** — Synthetic Minority Over-sampling Technique
- **RandomUnderSampler** — random removal of majority class samples
- **SMOTE + TomekLinks** — oversampling followed by Tomek Links cleaning

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate

# Classificadores
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Pipeline com suporte a samplers (imblearn)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

import warnings
warnings.filterwarnings('ignore')

## 2. Load and prepare data

In [ ]:
# Load encoded dataset
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, not ckd=1 (or vice versa)

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Reusable Pipeline

The `make_pipeline` function encapsulates:
1. **`MinMaxScaler`** — scaling fitted only on the training data of each fold  
2. **Sampler (SMOTE by default)** — balancing applied only to the training data  
3. **Classifier** — interchangeable without changing the rest of the pipeline  

> `imblearn.pipeline.Pipeline` is used instead of `sklearn.pipeline.Pipeline` to support samplers (which change the number of samples via `fit_resample`).

In [ ]:
def make_pipeline(classifier, sampler=SMOTE(random_state=42)):
    steps = [
        ("scaler", MinMaxScaler()),
        ("classifier", classifier),
    ]
    if sampler is not None:
        steps.insert(1, ("sampler", sampler))
    return Pipeline(steps)

## 4. Algorithms Definition

In [ ]:
models = {
    "KNN":      KNeighborsClassifier(),
    "DTree":    DecisionTreeClassifier(random_state=42),
    "RF":       RandomForestClassifier(random_state=42),
    "SVM":      SVC(random_state=42),
    "NB":       GaussianNB(),
}

## 5. Balancing Strategies

In [ ]:
strategies = {
    "No Balancing":       None,
    "SMOTE":              SMOTE(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "SMOTE + TomekLinks": SMOTETomek(random_state=42),
}

## 6. Spot-Checking with K-Fold Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1']

# all_results[strategy][model] = cross_validate scores dict
all_results = {}

for strategy_name, sampler in strategies.items():
    all_results[strategy_name] = {}
    for model_name, model_candidate in models.items():
        pipeline = make_pipeline(model_candidate, sampler=sampler)
        scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
        all_results[strategy_name][model_name] = scores

# Build summary DataFrame (mean and std per metric, strategy and model)
rows = []
for strategy_name, model_results in all_results.items():
    for model_name, scores in model_results.items():
        row = {"Strategy": strategy_name, "Model": model_name}
        for metric in scoring:
            row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
            row[f"{metric}_std"]  = np.std(scores[f"test_{metric}"])
        rows.append(row)

summary_df = (
    pd.DataFrame(rows)
    .sort_values(["Strategy", "f1_mean"], ascending=[True, False])
    .set_index(["Strategy", "Model"])
)

display(summary_df)

## 7. Boxplot (CV)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (strategy_name, model_results) in zip(axes, all_results.items()):
    f1_data = {
        model_name: scores["test_f1"].tolist()
        for model_name, scores in model_results.items()
    }
    pd.DataFrame(f1_data).boxplot(ax=ax)
    ax.set_title(strategy_name)
    ax.set_ylabel("F1-score")
    ax.set_ylim(0, 1.05)

plt.suptitle("F1-score Distribution per Balancing Strategy (CV)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()